this note book contains the cleaning and processing for the incident records

# 1.0 Load packages

In [ ]:
# load the packages
import pandas as pd
import json
import math
import numpy as np
from scipy.spatial.distance import cdist
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from itertools import cycle

In [ ]:
# load the incident data
incident_df = pd.read_csv('src_BD_IncidentData.csv')

In [ ]:
# Load schools
school_df = pd.read_csv('results.csv')

In [ ]:
# Load UK boundary
uk_boundary = gpd.read_file('boundary.geojson')

# 2.0 Data Preview

## 2.1 Incident data

In [ ]:
# View incident data
incident_df.head(3)

In [ ]:
# View the shape
incident_df.shape

In [ ]:
# Check data content
incident_df.info()

In [ ]:
# check description of entries
incident_df.describe()

Easting and Northing with 1.0 zero is a red flag

In [ ]:
# check columns
incident_df.columns

In [ ]:
# check the unique entries
incident_df.initial_grade.unique()

In [ ]:
incident_df.final_grade.unique()

In [ ]:
incident_df.qualifiers.unique()

These are not clear; would require metadeta for clarity.

In [ ]:
incident_df.reswodep.unique()

In [ ]:
incident_df.postcode_district.unique()

In [ ]:
incident_df.lsoa.unique()

In [ ]:
incident_df.lsoa_name.nunique()

In [ ]:
incident_df.lsoa_name.unique()

In [ ]:
incident_df.cb_date_added.unique()

In [ ]:
# inspect null conentions
for col in incident_df.columns:
    print('--------------------------------------------')
    unique_vals = incident_df[col].unique()
    print(f'{col}:')
    print(f'{unique_vals}')
    print(f'Total Values: {len(unique_vals)}')

## 2.2 School Data

In [ ]:
# View school data
school_df

In [ ]:
# view thw shape of the data
school_df.shape

In [ ]:
# check teh datatypes
school_df.info()

In [ ]:
school_df.columns

In [ ]:
# change name
school_df = school_df.rename(columns={'EstablishmentName': 'school_name'})

## 2.3 Boundary

In [ ]:
# view boundary data
uk_boundary

In [ ]:
# Filter for bradford
bradford_boundary = uk_boundary[
uk_boundary['LSOA21NM'].str.contains('Bradford', case=False, na=False)
]

In [ ]:
bradford_boundary.shape

In [ ]:
bradford_boundary.info()

In [ ]:
bradford_boundary.crs

# 4.0 Data Cleaning

## 4.1 Boundary

In [ ]:
# Check remaining columns
bradford_boundary.columns

In [ ]:
# Drop some other columns
bradford_boundary = bradford_boundary.drop(
    ['FID', 'BNG_E', 'LSOA21NMW', 'BNG_N', 'GlobalID'], axis=1)

In [ ]:
# Rename the columns
bradford_boundary = bradford_boundary.rename(
    columns={
        'LSOA21NMW': 'LSOA name',
        'LONG': 'bound_long',
        'LAT': 'bound_lat'}
)

In [ ]:
# Check coordinates
print(bradford_boundary.crs)

In [ ]:
# Transform to British National Grid (Meters)
bradford_boundary = bradford_boundary.to_crs(epsg=27700)

# confoirm change
print(bradford_boundary.crs)

## 4.2 School data

In [ ]:
# convert to time format
school_df['CloseDate'] = pd.to_datetime(school_df['CloseDate'], format='%d-%m-%Y', errors='coerce')

In [ ]:
# define cutoff 
cutoff_close = pd.to_datetime('2016-12-31', format='%Y-%m-%d')

# filter the cutoff
school_df = school_df[(school_df['CloseDate'] > cutoff_close) | (school_df['CloseDate'].isna())].copy()

In [ ]:
# remove duplicate schools using locations
school_df = school_df.drop_duplicates(subset=['Easting', 'Northing'], keep='last').copy()

In [ ]:
# Filter the cutoff: Keep schools closing AFTER the cutoff OR schools that are still open
school_df = school_df[(school_df['CloseDate'] > cutoff_close) | (school_df['CloseDate'].isna())].copy()

In [ ]:
school_df.school_name.nunique()

In [ ]:
school_df['SchoolCapacity'] = pd.to_numeric(school_df['SchoolCapacity'], errors='coerce')

In [ ]:
# replace nan on the schoolcapacity
cap_med = school_df['SchoolCapacity'].median()
school_df['SchoolCapacity'] = school_df['SchoolCapacity'].fillna(cap_med)

In [ ]:
# Drop the unnecessary columns
school_df = school_df.drop(
    ['UKPRN', 'Trusts (name)', 'SchoolSponsors (name)', 'CensusDate', 'FEHEIdentifier','SchoolWebsite',
     'InspectorateReport', 'DateOfLastInspectionVisit', 'LSOA (name)', 'PropsName', 'UPRN',
     'PropsType (name)', 'LSOA (code)', 'FurtherEducationType (name)', 'EstablishmentStatus (name)', 'OpenDate',
    'ReasonEstablishmentClosed (name)', 'CloseDate', 'PhaseOfEducation (name)', 'NumberOfPupils'],
    axis=1)

In [ ]:
# convert easting and northing to meter
geometry=gpd.points_from_xy(
        school_df['Easting'], school_df['Northing'])

# transform to british grid
sec_gdf = gpd.GeoDataFrame(school_df, geometry=geometry, crs="EPSG: 27700")

In [ ]:
# get the unique names
unique_names = sec_gdf['school_name'].unique()

# initialise empty dictionary
anon = {}

# loop for each school to create a schxx name
for idx, name in enumerate(unique_names):
    anon[name] = f"school{idx+1:02d}"

In [ ]:
# map it to your dataframe column
sec_gdf['sch_id'] = sec_gdf['school_name'].map(anon)

## 4.3 Incident data

In [ ]:
# view thw unique incidents
incident_df['incident_type'].unique()

In [ ]:
# reduce to lower characters
incident_df['incident_type'] = incident_df['incident_type'].str.lower()

I will select only incidences with major occurence.

In [ ]:
#check the percentage of each to 
group_df = incident_df.groupby('incident_type').size().reset_index(name='count')
group_df['percentage'] = (group_df['count'] / group_df['count'].sum()) * 100

# sort
group_df = group_df.sort_values(by='count', ascending=False).reset_index(drop=True)

In [ ]:
# select only the 1000+ incidents
interest = group_df[group_df['count'] >= 1000]

In [ ]:
# view what is left
interest = interest['incident_type'].unique()
interest

In [ ]:
# select only the interest inicidences
incident_df = incident_df[incident_df['incident_type'].isin(interest)].copy()

In [ ]:
# drop unnecessary columns
incident_df = incident_df.drop(
    ['Unnamed: 0', 'cb_date_added', 'post_code', 'lsoa', 'lsoa_name', 'incident_sub_type', 'postcode_district',
     'disposition_1_code',  'reswodep', 'qualifiers', 'Index', 'initial_grade', 'final_grade', 'attended',
    'incident_end_date'], axis=1)

In [ ]:
# rename columns
incident_df = incident_df.rename(
    columns={'lsoa1': 'lsoa',
             'lsoa_name1': 'lsoa_name',
             'incident_start_date': 'date_earliest',
             'incident_type': 'incident'
            }
)

In [ ]:
# remove the duplicates
incident_df = incident_df.drop_duplicates(subset=['incident_ref'])

### 4.3.1 Handle missing values

In [ ]:
# check the null values
incident_df.isnull().sum()

In [ ]:
# check minimum values
incident_df.min()

In [ ]:
# remove the row with 1.0 as goemetry
incident_df = incident_df[incident_df['easting'] != 1.0]

### 4.3.2 Remove incidents outside bradford

In [ ]:
# convert the crime data to geodataframe and transform to british national grid 
# creat geomerty point
geometry=gpd.points_from_xy(
        incident_df['easting'], incident_df['northing'])

# transform to british grid
incident_gdf = gpd.GeoDataFrame(incident_df, geometry=geometry, crs="EPSG: 27700")

In [ ]:
# check the grid
print(incident_gdf.crs)

In [ ]:
# drop everything outside the Bradford shape boundary
incident_gdf = gpd.clip(incident_gdf, bradford_boundary)

In [ ]:
# convert the datatime to its format
incident_gdf['date_earliest'] = pd.to_datetime(
    incident_gdf['date_earliest'], errors='coerce')

### 4.3.3 Derive new columns

In [ ]:
incident_gdf['date_earliest'].min()

In [ ]:
# extract the year, month and day
incident_gdf['year'] = incident_gdf['date_earliest'].dt.year
incident_gdf['month'] = incident_gdf['date_earliest'].dt.month
incident_gdf['day'] = incident_gdf['date_earliest'].dt.day
incident_gdf['date_only'] = incident_gdf['date_earliest'].dt.date

In [ ]:
# add name for month and day
incident_gdf['month_name'] = incident_gdf['date_earliest'].dt.month_name()
incident_gdf['day_name'] = incident_gdf['date_earliest'].dt.day_name()

In [ ]:
# add month label
incident_gdf["month_label"] = (
    incident_gdf["year"].astype(str)
    + "-"
    + incident_gdf["month"].astype(str).str.zfill(2)
)

In [ ]:
# extract time
incident_gdf['time'] = incident_gdf['date_earliest'].dt.strftime('%H:%M')

In [ ]:
# convert the hhmm time to minutes
def hhmm_to_minutes(value):
    if pd.isna(value):
        return None

    value = str(value)

    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)

In [ ]:
# map with times
incident_gdf['time_min'] = incident_gdf['time'].apply(hhmm_to_minutes)

In [ ]:
# create the day of week type
def classify_day_type(day):
    if day in ['Sunday', 'Saturday']:
        return 'Weekend'
    else:
        return 'Weekday'

In [ ]:
# map the data
incident_gdf['day_type'] = incident_gdf['day_name'].apply(classify_day_type)

In [ ]:
# keep only weekday crimes
incident_gdf = incident_gdf[~incident_gdf['day_name'].isin(['Saturday', 'Sunday'])].copy()

In [ ]:
# map the order of the days
day_map = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

incident_gdf['day_num'] = incident_gdf['day_name'].str.strip().map(day_map)

In [ ]:
# calculate the nearest hour withi which crime happened
incident_gdf['hour'] = incident_gdf['time_min'] // 60

# add 24 hours to the hour after each day
incident_gdf['weekly_hour'] = (incident_gdf['day_num'] * 24) + incident_gdf['hour']

In [ ]:
# define transition timing
def assign_commute(time_min):
    if 420 <= time_min < 540:
        return 'morning commute'
    elif 900 < time_min <= 1020:
        return 'afternoon commute'
    else:
        return 'non commute hours'

# map the column to time_min
incident_gdf['transition'] = incident_gdf['time_min'].apply(assign_commute)

In [ ]:
# map the column to time_min
incident_gdf = incident_gdf[incident_gdf['transition'].isin(['morning commute', 'afternoon commute'])]

In [ ]:
# Classify each date into the broad academic term
def classify_season_period(date):

    date_str = date.strftime('%Y-%m-%d')

    # 2017
    if '2017-01-01' <= date_str <= '2017-04-23':
        return 'spring'

    elif '2017-04-24' <= date_str <= '2017-08-31':
        return 'summer'

    elif '2017-09-01' <= date_str <= '2017-12-31':
        return 'autumn'

    # 2018
    elif '2018-01-01' <= date_str <= '2018-04-15':
        return 'spring'

    elif '2018-04-16' <= date_str <= '2018-08-31':
        return 'summer'

    elif '2018-09-01' <= date_str <= '2018-12-31':
        return 'autumn'

    # 2019
    elif '2019-01-01' <= date_str <= '2019-04-28':
        return 'spring'

    elif '2019-04-29' <= date_str <= '2019-08-31':
        return 'summer'

    elif '2019-09-01' <= date_str <= '2019-12-31':
        return 'autumn'

    # 2020
    elif '2020-01-01' <= date_str <= '2020-04-19':
        return 'spring'

    elif '2020-04-20' <= date_str <= '2020-08-31':
        return 'summer'

In [ ]:
# Create broad academic term column
incident_gdf['season'] = (
    incident_gdf['date_only']
    .apply(classify_season_period)
)

I will use Bingley Grammar School calendar for the holidays. See the reference:

2017/2018: https://bso.bradford.gov.uk/council/schools/holidaycalendarview.aspx?dfes=3805400&accyear=2017 \
2018/2019: https://bso.bradford.gov.uk/council/schools/holidaycalendarview.aspx?dfes=3805400&accyear=2018 \
2019/2020: https://bso.bradford.gov.uk/Schools/holidayCalendarView.aspx?accyear=2019&d=3805400

In [ ]:
# define school holiday periods
# Spring half-term - statutory days only
spring_half_term = [
    # 2017/18
    '2018-02-12', '2018-02-13', '2018-02-14',
    # 2018/19
    '2019-02-18', '2019-02-19', '2019-02-20',
    # 2019/20
    '2020-02-17', '2020-02-18', '2020-02-19'
]


# Easter
easter = [
    # 2017/18
    '2018-04-03', '2018-04-04', '2018-04-05', '2018-04-06', '2018-04-09', '2018-04-10', '2018-04-11', '2018-04-12', '2018-04-13',
    # 2018/19
    '2019-04-15', '2019-04-16', '2019-04-17', '2019-04-18', '2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26',
    # 2019/20
    '2020-04-06', '2020-04-07', '2020-04-08', '2020-04-09', '2020-04-14', '2020-04-15', '2020-04-16', '2020-04-17'
]

# Summer half-term - statutory days only
summer_half_term = [
    # 2017/18
    '2018-05-28', '2018-05-29', '2018-05-30', '2018-05-31', '2018-06-01',
    # 2018/19
    '2019-05-28', '2019-05-29', '2019-05-30', '2019-05-31',
    # 2019/20
    '2020-05-26', '2020-05-27', '2020-05-28', '2020-05-29'
]

# Summer holiday
summer = [
    # Summer 2017
    '2017-08-01', '2017-08-02', '2017-08-03', '2017-08-04', '2017-08-07', '2017-08-08', '2017-08-09', '2017-08-10', '2017-08-11',
    '2017-08-14', '2017-08-15', '2017-08-16', '2017-08-17', '2017-08-18', '2017-08-21', '2017-08-22', '2017-08-23', '2017-08-24',
    '2017-08-25', '2017-08-29', '2017-08-30', '2017-08-31', '2017-09-01',
    # Summer 2018
    '2018-07-26', '2018-07-27', '2018-07-30', '2018-07-31', '2018-08-01', '2018-08-02', '2018-08-03', '2018-08-06', '2018-08-07',
    '2018-08-08', '2018-08-09', '2018-08-10', '2018-08-13', '2018-08-14', '2018-08-15', '2018-08-16', '2018-08-17', '2018-08-20',
    '2018-08-21', '2018-08-22', '2018-08-23', '2018-08-24', '2018-08-28', '2018-08-29', '2018-08-30', '2018-08-31',
    # Summer 2019
    '2019-07-22', '2019-07-23', '2019-07-24', '2019-07-25', '2019-07-26', '2019-07-29', '2019-07-30', '2019-07-31', '2019-08-01',
    '2019-08-02', '2019-08-05', '2019-08-06', '2019-08-07', '2019-08-08', '2019-08-09', '2019-08-12', '2019-08-13', '2019-08-14',
    '2019-08-15', '2019-08-16', '2019-08-19', '2019-08-20', '2019-08-21', '2019-08-22', '2019-08-23', '2019-08-27', '2019-08-28',
    '2019-08-29', '2019-08-30',

    # Summer 2020
    '2020-07-21', '2020-07-22', '2020-07-23', '2020-07-24', '2020-07-27', '2020-07-28', '2020-07-29', '2020-07-30', '2020-07-31'
]

# Autumn half-term - statutory days only
autumn_half_term = [
    # 2017/18
    '2017-10-20', '2017-10-23', '2017-10-24', '2017-10-25',
    # 2018/19
    '2018-10-22', '2018-10-23', '2018-10-24',
    # 2019/20
    '2019-10-28', '2019-10-29', '2019-10-30'
]

# Christmas - statutory days only
christmas = [
    # 2017/18
    '2017-12-21', '2017-12-22', '2017-12-27', '2017-12-28', '2017-12-29', '2018-01-02', '2018-01-03',
    # 2018/19
    '2018-12-24', '2018-12-27', '2018-12-28', '2018-12-31', '2019-01-02', '2019-01-03',
    # 2019/20
    '2019-12-23', '2019-12-24', '2019-12-27', '2019-12-30', '2019-12-31', '2020-01-02', '2020-01-03'
]

# Training / INSET days
inset_days = [
    # 2017/18
    '2017-09-04', '2017-09-05', '2017-09-27', '2018-02-09', '2018-03-29',
    # 2018/19
    '2018-09-03', '2018-09-04', '2018-09-20', '2019-01-04', '2019-04-12',
    # 2019/20
    '2019-09-02', '2019-09-03', '2019-09-25', '2020-02-14', '2020-07-20'
]

# Bank holidays shown on the Bingley calendars
bank_holidays = [
    # 2017/18
    '2017-08-28', '2017-12-25', '2017-12-26', '2018-01-01', '2018-03-30', '2018-04-02', '2018-05-07', '2018-08-27',
    # 2018/19
    '2018-12-25', '2018-12-26', '2019-01-01', '2019-04-19', '2019-04-22', '2019-05-06', '2019-05-27', '2019-08-26',
    # 2019/20
    '2019-12-25', '2019-12-26', '2020-01-01', '2020-04-10', '2020-04-13', '2020-05-08', '2020-05-25'
]

# Occasional days
occasional_days = [
    # 2017/18
    '2017-10-26', '2017-10-27', '2018-02-15', '2018-02-16',
    # 2018/19
    '2018-10-25', '2018-10-26', '2019-02-21', '2019-02-22',
    # 2019/20
    '2019-10-31', '2019-11-01', '2020-02-20', '2020-02-21'
]

In [ ]:
# Classify weekdays by school-calendar period
def classify_school_period(date_val):

    # Exclude weekends completely
    if date_val.weekday() >= 5:
        return pd.NA

    date_str = date_val.strftime('%Y-%m-%d')

    if date_str in occasional_days:
        return 'occasional days'

    if date_str in autumn_half_term:
        return 'autumn half term'

    elif date_str in christmas:
        return 'christmas'

    elif date_str in spring_half_term:
        return 'Spring half term'

    elif date_str in easter:
        return 'easter'

    elif date_str in summer_half_term:
        return 'summer half term'

    elif date_str in summer:
        return 'summer'

    elif date_str in inset_days:
        return 'inset days'

    elif date_str in bank_holidays:
        return 'bank holiday'

    else:
        return 'Term time'

In [ ]:
# Map the school-period classification across all dates
incident_gdf['term_time_subperiod'] = (
    incident_gdf['date_only']
    .apply(classify_school_period)
)

In [ ]:
# default everything to holiday first
incident_gdf['term_time_type'] = 'holiday'

# overwrite term time
incident_gdf.loc[incident_gdf['term_time_subperiod'] == 'Term time', 'term_time_type'] = 'Term time'

In [ ]:
# create and get the week number
incident_gdf['week_number'] = pd.to_datetime(incident_gdf['date_only']).dt.isocalendar().week
incident_gdf['week_padded'] = incident_gdf['week_number'].astype(str).str.zfill(2)

In [ ]:
# pad the year with the week
incident_gdf['year_label'] = pd.to_datetime(incident_gdf['date_only']).dt.isocalendar().year
incident_gdf['year&week'] = incident_gdf['year_label'].astype(str) + "-W" + incident_gdf['week_padded']

In [ ]:
# add school and holiday suffixes to specify days within school week and holiday weeks
# copy the year&week
incident_gdf['year_week'] = incident_gdf['year&week']

# add 's' to school days and 'h' to holidays
incident_gdf.loc[incident_gdf['term_time_type'] == 'school day', 'year_week'] += 's'
incident_gdf.loc[incident_gdf['term_time_type'] != 'school day', 'year_week'] += 'h'

# 5.0 Sample selection

In [ ]:
# view the crimes with years
incident_gdf['year'].value_counts().to_frame()

I will use all the avaible dataset. This should be sufficient.

In [ ]:
incident_gdf = incident_gdf[incident_gdf['transition'] != 'non commute hours'].copy()

In [ ]:
incident_gdf.date_only.min()

In [ ]:
# save the cleaned crime data
incident_gdf.to_file('incident_gdf.geojson', driver='GeoJSON', index=False)

In [ ]:
# join the school with the distance
incident_nearest_school = gpd.sjoin_nearest(
    incident_gdf, sec_gdf, how="left",
    exclusive=True,
    distance_col="distance_to_school_m"
)

# 6.0 Exploratory Data Analysis

## 6.1 Overview

In [ ]:
# set plot layout
fig, ax = plt.subplots(figsize=(12, 10))

# lay the boundaries
bradford_boundary.plot(ax=ax, color='#f2f2f2', edgecolor='#b2b2b2', linewidth=1)

# plot crime points
incident_nearest_school.plot(ax=ax, color='#2C3E50', markersize=1,
                 alpha=0.6, zorder=2, label='Recorded Incident')

# lay the school points
sec_gdf.plot(ax=ax, color='#DC143C', markersize=12,
             marker='^', zorder=3, label='Secondary School')

# labels and titles
ax.set_axis_off()
plt.title("Crime Incidents", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 6.2 Univariate analysis

In [ ]:
# set up side-by-side plotting area
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Spatial Proximity distribution
sns.histplot(incident_nearest_school["distance_to_school_m"], bins=20, kde=True, ax=axes[0], color="#D62728")
axes[0].set_title("Distribution of Distance to Nearest School")
axes[0].set_xlabel("Distance (meters)")

# Plot Commute Minutes
sns.histplot(incident_nearest_school["time_min"], bins=20, kde=True, ax=axes[1], color="#1F77B4")
axes[1].set_title("Distribution of Commute Incident Times")
axes[1].set_xlabel("Time of Day (Minutes past midnight)")

plt.tight_layout()
plt.show()

In [ ]:
# Variables relevant to your filtered commute dataset
commute_variables = ["term_time_type", "day_name", "season", "month_name", "year_label"]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, var in enumerate(commute_variables):
    cat_order = incident_nearest_school[var].value_counts().index
    
    sns.countplot(x=var, 
        data=incident_nearest_school, ax=axes[i], order=cat_order, 
        hue=var, palette="Blues_r", legend=False)
    
    axes[i].set_title(f"Distribution of {var.replace('_', ' ').title()}")
    axes[i].tick_params(axis="x", rotation=45)

# Remove the unused space
fig.delaxes(axes[-1])
fig.suptitle("Commute Time Window Characteristics (Mon-Fri, 7-9AM & 3-5PM)", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# remove the index
plot_data = incident_nearest_school.reset_index(drop=True)

# set the columns
indicator_cols = ["transition", "year_week", "week_padded"]

In [ ]:
# Create one subplot for each column
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, var in enumerate(indicator_cols):
    # Show only the 10 most frequent categories
    top_categories = plot_data[var].value_counts().head(10).index

    # Count incidents in each category
    sns.countplot(data=plot_data, x=var,order=top_categories,hue=var,
                  palette="viridis",legend=False,ax=axes[i])

    # Format this subplot
    axes[i].set_title(f"Top Categories: {var.replace('_', ' ').title()}")
    axes[i].set_xlabel(var)
    axes[i].set_ylabel("Incident Count")
    axes[i].tick_params(axis="x", rotation=45)

# Add the overall title and adjust spacing
fig.suptitle(
    "Analysis of Engineered Time Windows and Suffix Profiles",
    fontsize=16,
    fontweight="bold"
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# select spatial and hourly markers inside the commute periods
numerical_metrics = ["distance_to_school_m", "hour", "weekly_hour"]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.flatten()

for i, var in enumerate(numerical_metrics):
    sns.histplot(data=incident_nearest_school, x=var, kde=True, ax=axes[i], color="#1F77B4")
    axes[i].set_title(var.replace("_", " ").title())
    axes[i].set_ylabel("Incident Frequency")

fig.suptitle("Spatial and Discrete Hour Commute Footprints", fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 6.3 Bivariate analysis

In [ ]:
plt.figure(figsize=(10, 5))

# Compares the exact proximity spread across your transition slots
sns.boxplot(data=incident_nearest_school, x='transition', 
            y='distance_to_school_m', hue='transition',
            palette=['#D62728', '#1F77B4'], legend=False)

plt.title('School Distance Proximity Compared Across Commute Windows', fontweight='bold')
plt.xlabel('Commute Window')
plt.ylabel('Distance to School (Meters)')
plt.tight_layout()
plt.show()

In [ ]:
# Matrix of day names versus school operation status
day_term_matrix = pd.crosstab(
    index=incident_nearest_school['day_name'], 
    columns=incident_nearest_school['term_time_type'],
    normalize='columns'
) * 100

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(day_term_matrix, annot=True,
            fmt=".1f", cmap="YlGnBu",
            cbar_kws={'label': '% Share'})
plt.title('Weekday Crime Distribution: School Term vs Holiday (%)', fontweight='bold')
plt.ylabel('Day of the Week')
plt.xlabel('Academic Status')
plt.tight_layout()
plt.show()

## Save all data for the next notebook

In [ ]:
# save all the data
df = incident_nearest_school.copy()
if 'time' in df.columns:
    df['time'] = df['time'].astype(str)
    
df.to_parquet('incident_nearest_school.parquet', index=False)
df.to_file('incident_nearest_school.geojson', driver='GeoJSON', index=False)